In [4]:
import pandas as pd

colunas_manter = [
    # Filtros temporários
    'IN_TREINEIRO', 'TP_ST_CONCLUSAO',
    'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
    'TP_STATUS_REDACAO',
    # Perfil Demográfico e Escolar
    'TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ENSINO',
    'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC',
    # Questionário Socioeconômico (Q001 a Q025)
    'Q001', 'Q002', 'Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010',
    'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 'Q019', 'Q020',
    'Q021', 'Q022', 'Q023', 'Q024', 'Q025',
    # Notas para cálculo da média
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO'
]

blocos_limpos = []

for chunk in pd.read_csv('MICRODADOS_ENEM_2023.csv', sep=';', encoding='latin1', usecols=colunas_manter, chunksize=200000):
    filtro = (
        (chunk['IN_TREINEIRO'] == 0) &
        (chunk['TP_ST_CONCLUSAO'] == 2) &  # Apenas concluintes cursando o 3º ano em 2023
        (chunk['TP_PRESENCA_CN'] == 1) &
        (chunk['TP_PRESENCA_CH'] == 1) &
        (chunk['TP_PRESENCA_LC'] == 1) &
        (chunk['TP_PRESENCA_MT'] == 1) &
        (chunk['TP_STATUS_REDACAO'] == 1)
    )
    chunk_filtrado = chunk[filtro].copy()

    # Apagar colunas de controle do filtro
    colunas_filtro = [
        'IN_TREINEIRO', 'TP_ST_CONCLUSAO',
        'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
        'TP_STATUS_REDACAO'
    ]
    chunk_filtrado.drop(columns=colunas_filtro, inplace=True)
    blocos_limpos.append(chunk_filtrado)

df = pd.concat(blocos_limpos, ignore_index=True)

# Remover linhas residuais de alunos do 3º ano que não tiveram a escola vinculada no INEP
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total de alunos do 3º ano com dados 100% completos: {len(df)}")
print("Contagem de nulos restante:\n", df.isna().sum().sum())

Total de alunos do 3º ano com dados 100% completos: 681357
Contagem de nulos restante:
 0


In [5]:
# 1. Calcular a Nota Média temporariamente apenas para achar o corte do 1º Quartil
colunas_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
nota_media_temp = df[colunas_notas].mean(axis=1)

# 2. Definir a variável-alvo única: RISCO_DEFASAGEM (1 = Risco / 1º Quartil, 0 = Adequado)
limiar_defasagem = nota_media_temp.quantile(0.25)
df['RISCO_DEFASAGEM'] = (nota_media_temp < limiar_defasagem).astype('int8')

print(f"Nota de corte para Risco de Defasagem (1º Quartil): {limiar_defasagem:.2f}")

# 3. Apagar as 5 notas individuais (já consolidadas na média)
df.drop(columns=colunas_notas, inplace=True)

# 4. (Opcional, mas recomendado) Apagar itens obsoletos para encurtar o formulário do gestor
itens_obsoletos = ['Q015', 'Q017', 'Q020', 'Q023'] # Secadora, Lava-louças, DVD e Telefone Fixo
df.drop(columns=itens_obsoletos, inplace=True)

Nota de corte para Risco de Defasagem (1º Quartil): 491.68


In [6]:
# 1. Variável Binária: TP_SEXO ('M' -> 0, 'F' -> 1)
df['TP_SEXO'] = df['TP_SEXO'].map({'M': 0, 'F': 1}).astype('int8')

# 2. Escolaridade dos Pais (Q001 e Q002): A (0) até G (6). 'H' (Não sei) vira -1
mapa_escolaridade_pais = {
    'H': -1, 'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6
}
df['Q001'] = df['Q001'].map(mapa_escolaridade_pais).astype('int8')
df['Q002'] = df['Q002'].map(mapa_escolaridade_pais).astype('int8')

# 3. Ocupação dos Pais (Q003 e Q004): A (0) até E (4). 'F' (Não sei) vira -1
mapa_ocupacao_pais = {
    'F': -1, 'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4
}
df['Q003'] = df['Q003'].map(mapa_ocupacao_pais).astype('int8')
df['Q004'] = df['Q004'].map(mapa_ocupacao_pais).astype('int8')

# 4. Renda Familiar (Q006): Ordinal de 'A' (0) até 'Q' (16)
letras_renda = list('ABCDEFGHIJKLMNOPQ')
mapa_renda = {letra: idx for idx, letra in enumerate(letras_renda)}
df['Q006'] = df['Q006'].map(mapa_renda).astype('int8')

# 5. Bens e Conectividade (Ordinais de A=0 até E=4)
colunas_bens = [
    'Q009', 'Q014', 'Q019', 'Q022',                 # Mantidas no Top 15
    'Q010', 'Q011', 'Q012', 'Q013', 'Q024', 'Q025'  # Temporárias para consolidação
]
mapa_bens = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
for col in colunas_bens:
    df[col] = df[col].map(mapa_bens).astype('int8')

# 6. Ajustar tipagem das colunas que já eram numéricas
for col in ['TP_FAIXA_ETARIA', 'Q005']:
    df[col] = df[col].astype('int8')

# 7. Engenharia de Atributos (Consolidação de Produto / UX em Binários 0 ou 1)
df['VEICULO_PROPRIO'] = ((df['Q010'] > 0) | (df['Q011'] > 0)).astype('int8')
df['REFRIGERACAO']    = ((df['Q012'] > 0) | (df['Q013'] > 0)).astype('int8')
df['CONECTIVIDADE']   = ((df['Q024'] > 0) | (df['Q025'] > 0)).astype('int8')

# Apagar as 6 colunas originais fragmentadas
df.drop(columns=['Q010', 'Q011', 'Q012', 'Q013', 'Q024', 'Q025'], inplace=True)

# 8. Organizar ordem final (15 Features + 1 Target = 16 Colunas) e Salvar
ordem_final = [
    'TP_FAIXA_ETARIA', 'TP_SEXO',
    'Q001', 'Q002', 'Q003', 'Q004', 'Q005', 'Q006',
    'Q009', 'Q014', 'Q019', 'Q022',
    'VEICULO_PROPRIO', 'REFRIGERACAO', 'CONECTIVIDADE',
    'RISCO_DEFASAGEM'
]
df = df[ordem_final]

df.to_parquet('ENEM_2023_3ANO_TOP15.parquet', index=False)
df.to_csv('ENEM_2023_3ANO_TOP15.csv', index=False, sep=';')

print("Pré-processamento concluído! Formato final da matriz:", df.shape)

Pré-processamento concluído! Formato final da matriz: (681357, 16)
